# Winger Shortlists (Club-Facing)

Build recruitment-ready winger shortlists by role from existing profile outputs.

In [6]:
from pathlib import Path
import pandas as pd

DATA_DIR = Path("../data/processed")
scouting_path = DATA_DIR / "winger_scouting_table.parquet"
stability_path = DATA_DIR / "winger_player_cluster_stability.parquet"

TOP_N = 50
MIN_STABILITY = 0.60


In [7]:
scouting = pd.read_parquet(scouting_path)
print("Scouting shape:", scouting.shape)

if stability_path.exists():
    stability = pd.read_parquet(stability_path)
    scouting = scouting.merge(
        stability[["player_id", "cluster_agreement"]],
        on="player_id",
        how="left",
    )
else:
    scouting["cluster_agreement"] = 1.0

scouting["cluster_agreement"] = scouting["cluster_agreement"].fillna(1.0)

required_pct = [
    "dribbles_per90_pct",
    "carries_per90_pct",
    "passes_per90_pct",
    "shots_per90_pct",
    "xg_per90_pct",
]
missing = [c for c in required_pct if c not in scouting.columns]
if missing:
    raise ValueError(f"Missing required percentile columns: {missing}")

scouting.head()


Scouting shape: (1438, 15)


,player_id,player_name,cluster,cluster_name,minutes,dribbles_per90,carries_per90,passes_per90,shots_per90,xg_per90,dribbles_per90_pct,carries_per90_pct,passes_per90_pct,shots_per90_pct,xg_per90_pct,cluster_agreement
0,5503,Lionel Andrés Messi Cuccittini,0,Inside Forward Finisher,22880.0,5.192308,47.934441,45.452360,3.666084,0.470774,97.913769,99.026426,97.844228,98.817803,98.400556,0.533333
1,4320,Neymar da Silva Santos Junior,0,Inside Forward Finisher,16510.0,6.045427,44.989098,40.393701,2.229558,0.376876,98.678720,98.331015,95.062587,92.280946,97.148818,0.533333
2,5477,Ousmane Dembélé,0,Inside Forward Finisher,10400.0,4.898077,38.682692,36.147115,1.929808,0.184544,97.218359,95.897079,90.820584,87.830320,84.561892,0.533333
3,3958,Pedro Eliezer Rodríguez Ledesma,0,Inside Forward Finisher,9880.0,2.186235,29.714575,28.566802,1.521255,0.189531,72.253129,83.101530,79.276773,77.955494,85.187761,0.533333
4,5487,Antoine Griezmann,0,Inside Forward Finisher,8450.0,1.192899,28.203550,33.209467,1.672189,0.239524,44.089013,79.694019,87.413074,82.127955,91.029207,0.533333


In [8]:
# Role-fit scores
s = scouting.copy()

s["score_direct_runner"] = (
    0.40 * s["dribbles_per90_pct"] +
    0.40 * s["carries_per90_pct"] +
    0.10 * s["shots_per90_pct"] +
    0.10 * s["xg_per90_pct"]
)

s["score_playmaker_winger"] = (
    0.55 * s["passes_per90_pct"] +
    0.20 * s["carries_per90_pct"] +
    0.15 * s["dribbles_per90_pct"] +
    0.10 * s["xg_per90_pct"]
)

s["score_inside_forward"] = (
    0.45 * s["shots_per90_pct"] +
    0.35 * s["xg_per90_pct"] +
    0.15 * s["dribbles_per90_pct"] +
    0.05 * s["carries_per90_pct"]
)

for col in ["score_direct_runner", "score_playmaker_winger", "score_inside_forward"]:
    s[f"{col}_adj"] = s[col] * s["cluster_agreement"]

role_scores = s
role_scores[["player_id", "cluster_name", "score_direct_runner_adj", "score_playmaker_winger_adj", "score_inside_forward_adj"]].head(10)


,player_id,cluster_name,score_direct_runner_adj,score_playmaker_winger_adj,score_inside_forward_adj
0,5503,Inside Forward Finisher,52.532221,52.344924,52.558183
1,4320,Inside Forward Finisher,52.131664,51.449235,50.798331
2,5477,Inside Forward Finisher,50.392211,49.157163,47.198887
3,3958,Inside Forward Finisher,41.843301,42.442281,42.607325
4,5487,Inside Forward Finisher,35.642096,42.523876,42.355123
5,15619,Inside Forward Finisher,47.528975,47.156236,46.510895
6,3385,Inside Forward Finisher,49.164580,47.582754,48.847473
7,3501,Inside Forward Finisher,43.341678,45.103384,44.914233
8,15570,Inside Forward Finisher,48.712100,44.000000,50.609179
9,2995,Inside Forward Finisher,49.910060,51.469634,47.616134


In [9]:
id_cols = ["player_id"] + (["player_name"] if "player_name" in role_scores.columns else []) + ["cluster", "cluster_name", "minutes", "cluster_agreement"]
metric_cols = [
    "dribbles_per90", "carries_per90", "passes_per90", "shots_per90", "xg_per90",
    "dribbles_per90_pct", "carries_per90_pct", "passes_per90_pct", "shots_per90_pct", "xg_per90_pct",
]

eligible = role_scores[role_scores["cluster_agreement"] >= MIN_STABILITY].copy()

shortlist_direct_runner = (
    eligible.sort_values("score_direct_runner_adj", ascending=False)
    [id_cols + metric_cols + ["score_direct_runner", "score_direct_runner_adj"]]
    .head(TOP_N)
    .reset_index(drop=True)
)

shortlist_playmaker_winger = (
    eligible.sort_values("score_playmaker_winger_adj", ascending=False)
    [id_cols + metric_cols + ["score_playmaker_winger", "score_playmaker_winger_adj"]]
    .head(TOP_N)
    .reset_index(drop=True)
)

shortlist_inside_forward = (
    eligible.sort_values("score_inside_forward_adj", ascending=False)
    [id_cols + metric_cols + ["score_inside_forward", "score_inside_forward_adj"]]
    .head(TOP_N)
    .reset_index(drop=True)
)

shortlist_direct_runner.head(10)


,player_id,player_name,cluster,cluster_name,minutes,cluster_agreement,dribbles_per90,carries_per90,passes_per90,shots_per90,xg_per90,dribbles_per90_pct,carries_per90_pct,passes_per90_pct,shots_per90_pct,xg_per90_pct,score_direct_runner,score_direct_runner_adj
0,6907,Pedro León Sánchez Gil,0,Inside Forward Finisher,4420.0,0.6,2.524887,20.789593,22.255656,1.099548,0.093182,80.111266,61.891516,63.838665,61.821975,57.927677,68.776078,41.265647
1,3083,Heung-Min Son,0,Inside Forward Finisher,3250.0,0.6,1.883077,19.107692,18.110769,1.661538,0.130544,66.342142,55.806676,50.417246,81.815021,71.210014,64.162031,38.497218
2,8233,Gonzalo Castro Randon,0,Inside Forward Finisher,1560.0,0.6,1.673077,26.653846,29.596154,0.634615,0.073220,60.361613,76.286509,81.432545,33.379694,48.956885,62.892907,37.735744


In [10]:
role_scores_out = DATA_DIR / "winger_role_scores.parquet"
shortlist_direct_out = DATA_DIR / "shortlist_direct_runner.csv"
shortlist_playmaker_out = DATA_DIR / "shortlist_playmaker_winger.csv"
shortlist_inside_out = DATA_DIR / "shortlist_inside_forward.csv"

role_scores.to_parquet(role_scores_out, index=False)
shortlist_direct_runner.to_csv(shortlist_direct_out, index=False)
shortlist_playmaker_winger.to_csv(shortlist_playmaker_out, index=False)
shortlist_inside_forward.to_csv(shortlist_inside_out, index=False)

print("Saved:")
print(" -", role_scores_out)
print(" -", shortlist_direct_out)
print(" -", shortlist_playmaker_out)
print(" -", shortlist_inside_out)


Saved:
 - ..\data\processed\winger_role_scores.parquet
 - ..\data\processed\shortlist_direct_runner.csv
 - ..\data\processed\shortlist_playmaker_winger.csv
 - ..\data\processed\shortlist_inside_forward.csv
